---
# 1. Load Packages

In [ ]:
# From a fresh python installation, install these:
# pip install notebook
# pip install holoviews
# pip install opencv-python
# pip install selenium
# pip install webdriver-manager
# pip install matplotlib
# pip install scipy

print("Importing libraries...")
import os
import holoviews as hv
import numpy as np
import pandas as pd
import LocationTracking_Functions as lt
from bokeh.io import output_notebook, show
import inspect
import time
import importlib
import socket
import tkinter as tk
from tkinter import filedialog
from pathlib import Path
import selenium  # This is needed by bokeh.io image export functions. Make sure you have latest version by running pip install -U selenium
from webdriver_manager.firefox import GeckoDriverManager      # Need to use pip install webdriver-manager to get this.
from selenium import webdriver

# Get the hostname of the local machine
hostname = socket.gethostname()
print(f"The machine's hostname is: {hostname}")

if hostname == "TJHomeOffice":
    PaulFolder = "D:\\University of Maryland School of Medicine\\Jhou Lab - General\\M_Rec\\"
    TeamsFolder = "D:\\University of Maryland School of Medicine\\JhouLab Overflow1 - General\\"
    RaidFolder = r"\\TOWER\share_zfs48_home/"
elif hostname == "TomOffice2025":
    PaulFolder = "D:\\University of Maryland School of Medicine\\Jhou Lab - General\\M_Rec\\"
    TeamsFolder = "D:/University of Maryland School of Medicine/JhouLab Overflow1 - General/"
    RaidFolder = r"\\LABUNRAID\zfs48_share/"
elif hostname == "TJ_gram":
    TeamsFolder = "C:/Users/tomjh/University of Maryland School of Medicine/JhouLab Overflow1 - General/"
    RaidFolder = r"\\LABUNRAID\zfs48_share/"
elif hostname == "DESKTOP-DIQ9828":
    RaidFolder = r"\\LABUNRAID\zfs48_share/"
else:
    print("Unrecognized machine, please choose top level video folder from dialog.")
    RaidFolder = r"\\LABUNRAID\zfs48_share/"

# Set gecko_driver environment variables
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager      # Need to use pip install webdriver-manager to get this
from pathlib import Path

# Automatically manage the geckodriver executable
service = FirefoxService(GeckoDriverManager().install())
gecko_driver_path = service.path

def find_firefox_windows():
    # Common default installation paths for Firefox on Windows
    # 64-bit Firefox on 64-bit Windows or 32-bit Firefox on 32-bit Windows
    paths = [
        Path("C:\\Program Files\\Mozilla Firefox\\firefox.exe"),
        # 32-bit Firefox on 64-bit Windows
        Path("C:\\Program Files (x86)\\Mozilla Firefox\\firefox.exe")
    ]    
    # Check for user-only installation via Microsoft Store or specific installer
    appdata_local = os.getenv("LOCALAPPDATA")
    if appdata_local:
        # The exact folder name for the App Store version can vary (e.g., Mozilla.Firefox_xxxxxxxxxxxxx)
        # This is a common location pattern for user-only installs
        paths.append(Path(appdata_local) / "Mozilla Firefox" / "firefox.exe")
    for path in paths:
        if path.is_file():
            print(f'Found Firefox installation at: "{path}"')
            return str(path)
    print('Unable to find firefox installation. Some graph generating code may not work.')
    return None

# Append firefox.exe to path
current_path = os.environ.get('PATH')
firefox_dir = os.path.dirname(find_firefox_windows())  # r"C:\Program Files (x86)\Mozilla Firefox/"   # Use "r" in front to prevent backslashes from being interpreted as escape symbols
if not current_path.endswith(os.pathsep):
    # Append semicolon if not already present. Usually it will already be there, so this is skipped
    current_path = current_path + os.pathsep
gecko_path = os.path.dirname(gecko_driver_path)
print(f'Found gecko installation at: "{gecko_path}"')

# Add firefox and gecko paths to PATH variable
os.environ['PATH'] = f"{current_path}{firefox_dir}{os.pathsep}{gecko_path}"
print(f'Added to environment PATH: {os.environ["PATH"]}')
print("Initialization completed.")

---
# 2. Choose file, Crop Frame

In [ ]:
%%output size = 200

## Prompt user to choose file from dialog box
root = tk.Tk()
root.withdraw()
root.wm_attributes('-topmost', True)

print('Please select AVI file from dialog box. Note: dialog might be behind the Python window, or on another screen.')

# relative_paths = [RaidFolder + "behavior_videos", RaidFolder + "behavior_videos_copy"]
relative_paths = [PaulFolder]

for x in relative_paths:
    if Path(x).is_dir():
        SrcFile = filedialog.askopenfilename(
            parent=root,
            initialdir = x,
            filetypes = (
                    ('AVI files', '*.avi'),   # Must have final comma, to make this a tuple
                )
        )
        break
root.destroy()
if SrcFile == '':
    print('No file selected')
else:
    print(f'You selected file "{SrcFile}"')
    video_dict = {
        'dpath'        : '',
        'file'         : SrcFile,
        'fpath'        : SrcFile,     # Full path
        'start'        : 0,
        'end'          : None,
        'num_animals'  : 1,        # How many animals are in this frame?
        'crop_names'   : ['animal1', 'animal2', 'animal3', 'animal4'],     # Name of each subject
        'region_names' : ['region1', 'region2', 'region3', 'region4'],
        'dsmpl'        : 0.5,           # Downsample proportion
        'stretch'      : dict(width=1, height=1)
    }

    tmp = Path(SrcFile).with_suffix("")
    fpath = Path(f"{tmp}_{str(video_dict['dsmpl'])}_Location.csv")
    CROP_NAME = ""
    img_crp, video_dict = lt.LoadAndCrop(video_dict, cropmethod='Box')
    display(img_crp)

In [ ]:

# Diagnostics ... prints all cropped regions
crop_dict = video_dict['crop'].data
for idx in range(video_dict['num_animals']):
    coords = [crop_dict['x0'][idx],
              crop_dict['x1'][idx],
              crop_dict['y0'][idx],
              crop_dict['y1'][idx]]
    print(f'Crop region: x={coords[0]} to {coords[1]}, y={coords[2]} to {coords[3]}\n')
    
    

---
### 3. Create reference frame, Track Location, save tracked video

In [ ]:
%%output size = 80

importlib.reload(lt)
video_dict['reference'] = []

# Make, show, and save reference image. This is done by removing mouse based on contrast with background
for idx in range(video_dict['num_animals']):
    ref, img_ref = lt.Reference(video_dict, num_frames=50, frames=None, crop_num=idx)
    video_dict['reference'].append(ref)
    fpath = lt.GetFileBase(video_dict) + "_" + video_dict['crop_names'][idx] + "_" + str(video_dict['dsmpl']) + "_reference"
    print(f'Saving to: {fpath}.png')
    hv.save(img_ref, fpath + ".png")
    hv.ipython.display(img_ref)
    print(f'Done')

tracking_params = {
    'loc_thresh'    : 95,   # Default percentile, can be overridden later if needed (but usually isn't)
    'use_window'    : False,
    'window_size'   : 150,
    'window_weight' : .9,
    'method'        : 'dark',
    'rmv_wire'      : True,
    'wire_krn'      : 2
}

# Show tracking examples
for idx in range(video_dict['num_animals']):
    img_exmpls = lt.LocationThresh_View(video_dict, tracking_params, examples=6, crop_num=idx)
    img_exmpls.cols(6)

    fpath = lt.GetFileBase(video_dict) + "_" + video_dict['crop_names'][idx] + "_" + str(video_dict['dsmpl']) + "_track_examples"
    print(f'Saving to file: {fpath}.png')
    hv.save(img_exmpls, fpath + ".png")
    hv.ipython.display(img_exmpls)
    print('Done saving')


# Track location
location = lt.TrackLocation(video_dict, tracking_params)
fpath = os.path.splitext(video_dict['fpath'])[0] + "_" + str(video_dict['dsmpl']) + '_Location.csv'

# Print stats
Dist = location['Dist_px0']
print(f"Mean distance per frame is: {Dist.mean():0.3f}, min is {Dist.min():0.3f}, max is {Dist.max():0.3f}")

print(f"Saving to file: {fpath}")
location.to_csv(fpath, index=False)
location.head()

#
# Show graphics and plots of movement
#
w, h = 600,200

fps = video_dict['nominal_fps']

for x in range(video_dict['num_animals']):
    Dist = location['Dist_px' + str(x)]
    plt_dist = hv.Curve((location['Frame'] / fps / 60, Dist), 'Time (minutes)', 'Pixel Distance').opts(
        height=h, width=w, color='red', title=f"Distance Across Session, mean={Dist.mean():0.3f}, SD={np.std(Dist):0.3f}", toolbar="below")
    plt_trks = lt.showtrace(video_dict, location, color="red", alpha=.05, size=2)
    plt_hmap = lt.Heatmap(video_dict, location, sigma=None)
    p = (plt_trks + plt_hmap + plt_dist).cols(3)
    # (plt + plt_dist)

    fpath = lt.GetFileBase(video_dict) + "_" + video_dict['crop_names'][x] + "_" + str(video_dict['dsmpl']) + "_movement.png"
    print(f'Saving to file: {fpath}')
    hv.save(p, fpath)
    hv.ipython.display(p)

display_dict = {
    'start'      : 0,   # If < video_dict['start'], will be coerced to that value
    'stop'       : None,    # If > video_dict['end'], will be coerced to that value
    'resize'     : None,
    'save_video' : True
}

start_time = time.time()
lt.PlayVideo(video_dict, display_dict, location)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.2f} seconds")


In [ ]:
importlib.reload(lt)
lt.PlayVideo(video_dict, display_dict, location)